In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast  # Mixed Precision for faster training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc, average_precision_score
import joblib
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# TPU/GPU DETECTION
# =============================================================================
print("\n" + "="*80)
print("HARDWARE DETECTION")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"Platform: {platform.platform()}")

# TPU setup
TPU_AVAILABLE = False
xm = None
xmp = None

try:
    import torch_xla
    import torch_xla.core.xla_model as xla_model
    TPU_AVAILABLE = True
    xm = xla_model
    print(f"✅ TPU Available: torch_xla {torch_xla.__version__}")
    try:
        import torch_xla.distributed.xla_multiprocessing as xla_mp
        xmp = xla_mp
    except ImportError:
        pass
except ImportError:
    print("⚠️ TPU not available (torch_xla not installed)")
    print("   For Kaggle TPU: Runtime → Change runtime type → TPU")

# CUDA setup
if torch.cuda.is_available():
    print(f"\n✅ CUDA Available")
    print(f"   GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB)")
else:
    print("\n⚠️ CUDA not available")

print("="*80 + "\n")

# Device selection
USE_XLA = False
DEVICE = None

if TPU_AVAILABLE and xm is not None:
    USE_XLA = True
    DEVICE = xm.xla_device()
    print(f"🎯 Using TPU: {DEVICE}")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🎯 Using CUDA: {DEVICE}")
else:
    DEVICE = torch.device("cpu")
    print(f"🎯 Using CPU: {DEVICE}")

print("")



# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset
DATA_DIR = "/kaggle/input/cic-ids2017/MachineLearningCVE"
CSV_FILES = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
]

# Output
OUT_DIR = "/kaggle/working/models"
PLOT_DIR = "/kaggle/working/plots"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Set seed for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 4096 if torch.cuda.is_available() else (1024 if TPU_AVAILABLE else 512)  # Increased for GPU speedup
LEARNING_RATE = 0.001
PATIENCE = 10


# =============================================================================
# MODEL SELECTION - Comment/uncomment to choose which models to train
# =============================================================================
MODELS_TO_TRAIN = {
    'mlp_model': True,          # ✅ Deep MLP (baseline) - Fast, ~5-8 min
    'cnn_model': True,          # ✅ 1D CNN - Moderate, ~8-12 min
    'tcn_model': True,          # ✅ Temporal CNN - Moderate, ~10-15 min
    'bilstm_model': True,       # ✅ BiLSTM-Attention - Moderate, ~10-15 min
}


# =============================================================================
# MODEL ARCHITECTURES (from ids.py)
# =============================================================================

class MLPDropout(nn.Module):
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.05),
            nn.Linear(64, d_out),
        )
    def forward(self, x): return self.net(x)


class DeepCNN(nn.Module):
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.proj = nn.Linear(d_in, 128)
        self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)  # Match saved checkpoint
        self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(64)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, d_out))
    def forward(self, x):
        x = F.relu(self.proj(x)).unsqueeze(1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        return self.fc(self.pool(x).squeeze(-1))


class DeepTCN(nn.Module):
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.input = nn.Linear(d_in, 64)
        self.tcn1 = nn.Conv1d(1, 32, 3, dilation=1, padding=1)
        self.bn1 = nn.BatchNorm1d(32)  # Match saved checkpoint
        self.tcn2 = nn.Conv1d(32, 64, 3, dilation=2, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.tcn3 = nn.Conv1d(64, 32, 3, dilation=4, padding=4)
        self.bn3 = nn.BatchNorm1d(32)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, d_out))
    def forward(self, x):
        x = F.relu(self.input(x)).unsqueeze(1)
        x = F.relu(self.bn1(self.tcn1(x)))
        x = F.relu(self.bn2(self.tcn2(x)))
        x = F.relu(self.bn3(self.tcn3(x)))
        return self.fc(self.pool(x).squeeze(-1))


class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with attention mechanism for IDS classification.
    
    References:
    - IntrusionX (Oct 2025): CNN-LSTM hybrid, 98% accuracy on NSL-KDD (arXiv:2510.00572)
    - xIDS-EnsembleGuard (Mar 2025): LSTM+GRU ensemble on CIC-IDS2017 (arXiv:2503.00615)
    - LSTM-CNN-Attention (Jan 2025): 99.04% accuracy on Edge-IIoTset (arXiv:2501.13962)
    - BiGRU-LSTM-Attention (Aug 2025): Cross-domain IoT security (arXiv:2508.12470)
    
    Architecture:
    - Input projection to reduce dimensionality
    - Bidirectional LSTM for temporal modeling (captures both past and future context)
    - Multi-head attention mechanism to focus on important features/timesteps
    - Dropout regularization to prevent overfitting
    - Output classification layer
    """
    def __init__(self, d_in: int, n_classes: int = 2, p: float = 0.2, hidden_size: int = 128, num_layers: int = 2, num_heads: int = 4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Input projection (reduce dimensionality for efficiency)
        self.input_proj = nn.Sequential(
            nn.Linear(d_in, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.Dropout(p)
        )
        
        # Bidirectional LSTM (captures temporal dependencies in both directions)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=p if num_layers > 1 else 0.0,
            bidirectional=True
        )
        
        # Multi-head attention mechanism (focuses on important timesteps/features)
        # Input to attention: hidden_size * 2 (bidirectional concatenation)
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size * 2,
            num_heads=num_heads,
            dropout=p,
            batch_first=True
        )
        
        # Layer normalization after attention
        self.attn_norm = nn.LayerNorm(hidden_size * 2)
        
        # Output classification layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch, features)
        
        Returns:
            logits: Output tensor of shape (batch, n_classes)
        """
        batch_size = x.size(0)
        
        # Project input to hidden dimension
        x = self.input_proj(x)  # (batch, hidden_size)
        
        # Reshape for LSTM: (batch, seq_len=1, hidden_size)
        # For tabular data, we treat each sample as a single timestep
        # The BiLSTM will still learn directional feature dependencies
        x = x.unsqueeze(1)  # (batch, 1, hidden_size)
        
        # Bidirectional LSTM
        # lstm_out: (batch, 1, hidden_size * 2) - concatenated forward and backward
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Apply self-attention
        # Query, Key, Value are all from lstm_out
        attn_out, attn_weights = self.attention(
            lstm_out, lstm_out, lstm_out
        )
        
        # Residual connection + layer normalization
        attn_out = self.attn_norm(lstm_out + attn_out)
        
        # Squeeze sequence dimension and classify
        attn_out = attn_out.squeeze(1)  # (batch, hidden_size * 2)
        
        # Classification
        logits = self.classifier(attn_out)  # (batch, n_classes)
        
        return logits

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================
print("\n" + "="*80)
print("DATA LOADING AND PREPROCESSING")
print("="*80)

# Metadata columns to drop
drop_columns = [
    "Flow ID",    
    'Fwd Header Length.1',
    "Source IP", "Src IP",
    "Source Port", "Src Port",
    "Destination IP", "Dst IP",
    "Destination Port", "Dst Port",
    "Timestamp",
]

# Load CSV files
print(f"Loading {len(CSV_FILES)} CSV files from {DATA_DIR}...")
individual_dfs = []
for idx, csv_file in enumerate(CSV_FILES, 1):
    file_path = os.path.join(DATA_DIR, csv_file)
    print(f"  [{idx}/{len(CSV_FILES)}] Loading: {csv_file}")
    df = pd.read_csv(file_path, sep=',', encoding='utf-8')
    print(f"      Initial shape: {df.shape}")
    print(f"      Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    individual_dfs.append(df)

print(f"\nTotal DataFrames loaded: {len(individual_dfs)}")
print(f"Total memory before optimization: {sum(df.memory_usage(deep=True).sum() for df in individual_dfs) / 1024**2:.2f} MB")

# Preprocess each dataframe
print("\nPreprocessing individual DataFrames...")
print("-" * 80)
for i, df in enumerate(individual_dfs):
    print(f"\nDataFrame {i} ({CSV_FILES[i]}):")
    
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
    print(f"  Total columns: {len(df.columns)}")
    
    # Drop metadata columns
    dropped = [col for col in drop_columns if col in df.columns]
    df.drop(columns=drop_columns, inplace=True, errors='ignore')
    if dropped:
        print(f"  Dropped metadata columns: {len(dropped)}")
    
    # Standardize label naming
    df['Label'] = df['Label'].replace({'BENIGN': 'Benign'})
    
    # Replace infinite values with NaN
    inf_count = np.isinf(df.select_dtypes(include=[np.number]).values).sum()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    if inf_count > 0:
        print(f"  Infinite values replaced with NaN: {inf_count}")
    
    # Drop NaN values
    nan_rows = df.isna().any(axis=1).sum()
    df.dropna(inplace=True)
    if nan_rows > 0:
        print(f"  Rows with NaN removed: {nan_rows}")
    
    # Drop duplicates
    duplicates = df.duplicated().sum()
    df.drop_duplicates(inplace=True)
    if duplicates > 0:
        print(f"  Duplicate rows removed: {duplicates}")
    
    df.reset_index(drop=True, inplace=True)
    print(f"  Final shape: {df.shape}")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Concatenate all dataframes
print("\nConcatenating DataFrames...")
df_data = pd.concat(individual_dfs, axis=0, ignore_index=True)
print(f"Combined dataset shape: {df_data.shape}")

# Additional cleaning
print("\nFinal data cleaning...")
null_counts = df_data.isnull().sum()
print(f"  Null entries found: {null_counts.sum()}")
if null_counts.sum() > 0:
    df_data.dropna(inplace=True)
    print(f"  After removing nulls: {df_data.shape}")

duplicate_count = df_data.duplicated().sum()
print(f"  Duplicate entries found: {duplicate_count}")
if duplicate_count > 0:
    df_data.drop_duplicates(inplace=True)
    print(f"  After removing duplicates: {df_data.shape}")

df_data.reset_index(drop=True, inplace=True)

# Inspect label distribution
print("\n" + "-"*80)
print("LABEL DISTRIBUTION (Before Binarization)")
print("-"*80)
print("\nAbsolute counts:")
print(df_data['Label'].value_counts().to_string())
print(f"\nRelative proportions:")
for label, prop in df_data['Label'].value_counts(normalize=True).items():
    print(f"  {label:<30} {prop*100:>6.2f}%")
print(f"\nTotal unique labels: {df_data['Label'].nunique()}")
print(f"Total samples: {len(df_data):,}")

# Extract features and target
print("\n" + "-"*80)
print("FEATURE EXTRACTION")
print("-"*80)
X = df_data.drop('Label', axis=1).copy()
y = df_data['Label'].copy()

# Binarize labels: Benign=0, Attack=1
print("\nBinarizing labels (Benign=0, Attack=1)...")
y = y.map({'Benign': 0}).fillna(1).astype(int)
print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"Features data type: {X.dtypes.value_counts().to_dict()}")
print(f"\nBinary class distribution:")
for class_label, count in pd.Series(y).value_counts().sort_index().items():
    class_name = "Benign" if class_label == 0 else "Attack"
    print(f"  Class {class_label} ({class_name}): {count:>10,} samples ({count/len(y)*100:>5.2f}%)")
print(f"\nClass imbalance ratio: {pd.Series(y).value_counts().max() / pd.Series(y).value_counts().min():.2f}:1")

# =============================================================================
# DATA SPLITTING
# =============================================================================
print("\n" + "="*80)
print("DATA SPLITTING")
print("="*80)

# Split: 75% train, 10% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    stratify=y,
    test_size=0.25,
    random_state=seed,
    shuffle=True
)

# Split temp into val and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    stratify=y_temp,
    test_size=0.6,  # 0.6 of 25% = 15% test
    random_state=seed,
    shuffle=True
)

print(f"\nDataset split results:")
print(f"  Train set: {X_train.shape[0]:>10,} samples ({X_train.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Val set:   {X_val.shape[0]:>10,} samples ({X_val.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Test set:  {X_test.shape[0]:>10,} samples ({X_test.shape[0]/len(X)*100:>5.2f}%)")
print(f"\nClass distribution in splits:")
print(f"  Train - Benign: {(y_train==0).sum():>10,}, Attack: {(y_train==1).sum():>10,}")
print(f"  Val   - Benign: {(y_val==0).sum():>10,}, Attack: {(y_val==1).sum():>10,}")
print(f"  Test  - Benign: {(y_test==0).sum():>10,}, Attack: {(y_test==1).sum():>10,}")

# =============================================================================
# DATA SCALING
# =============================================================================
print("\n" + "="*80)
print("DATA SCALING (Z-SCORE NORMALIZATION)")
print("="*80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train set scaled: {X_train_scaled.shape}")
print(f"Val set scaled:   {X_val_scaled.shape}")
print(f"Test set scaled:  {X_test_scaled.shape}")

# Save scaler for inference
scaler_path = os.path.join(OUT_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler saved: {scaler_path}")

# =============================================================================
# PYTORCH DATASET AND DATALOADER
# =============================================================================
print("\n" + "="*80)
print("CREATING PYTORCH DATASETS")
print("="*80)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train.values)
X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.LongTensor(y_val.values)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test.values)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True if torch.cuda.is_available() else False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True if torch.cuda.is_available() else False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True if torch.cuda.is_available() else False)

# Wrap dataloaders for TPU if available
if USE_XLA and xmp is not None:
    import torch_xla.distributed.parallel_loader as pl
    train_loader = pl.MpDeviceLoader(train_loader, DEVICE)
    val_loader = pl.MpDeviceLoader(val_loader, DEVICE)
    test_loader = pl.MpDeviceLoader(test_loader, DEVICE)
    print(f"\n🚀 TPU ParallelLoader enabled for optimized data transfer")

print(f"\nDataLoader configuration:")
print(f"  Batch size: {BATCH_SIZE} {'(TPU optimized)' if USE_XLA else ''}")
print(f"  Train batches: {len(train_loader):>6,} (last batch: {len(train_dataset) % BATCH_SIZE or BATCH_SIZE} samples)")
print(f"  Val batches:   {len(val_loader):>6,} (last batch: {len(val_dataset) % BATCH_SIZE or BATCH_SIZE} samples)")
print(f"  Test batches:  {len(test_loader):>6,} (last batch: {len(test_dataset) % BATCH_SIZE or BATCH_SIZE} samples)")
print(f"  Shuffle: Train=True, Val/Test=False")
print(f"  Pin memory: {torch.cuda.is_available()}")
if USE_XLA:
    print(f"  🚀 TPU Acceleration: ENABLED")
    print(f"     - Optimized batch size: {BATCH_SIZE}")
    print(f"     - XLA graph compilation: Active")
    print(f"     - ParallelLoader: Active")

# Calculate class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train.values
)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f"\nClass weights: {class_weights}")

# =============================================================================
# VISUALIZATION UTILITIES
# =============================================================================

def plot_training_history(train_history, val_history, model_name, save_path):
    """Plot training and validation loss/accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_history['loss']) + 1)
    
    # Loss plot
    axes[0].plot(epochs, train_history['loss'], 'b-', label='Training Loss', linewidth=2)
    axes[0].plot(epochs, val_history['loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title(f'{model_name} - Loss Curve', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy plot
    axes[1].plot(epochs, [acc*100 for acc in train_history['acc']], 'b-', label='Training Accuracy', linewidth=2)
    axes[1].plot(epochs, [acc*100 for acc in val_history['acc']], 'r-', label='Validation Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title(f'{model_name} - Accuracy Curve', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    📈 Training history plot saved: {save_path}")
    plt.close()


def plot_confusion_matrix(cm, model_name, save_path):
    """Plot confusion matrix heatmap."""
    plt.figure(figsize=(8, 6))
    
    # Normalize confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create annotations with both counts and percentages
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"
    
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', 
                xticklabels=['Benign', 'Attack'],
                yticklabels=['Benign', 'Attack'],
                cbar_kws={'label': 'Proportion'},
                linewidths=1, linecolor='gray')
    
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    📊 Confusion matrix plot saved: {save_path}")
    plt.close()


def plot_roc_curve(labels, probs, model_name, save_path):
    """Plot ROC curve and save with AUC annotation."""
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc*100:.2f}%)')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'{model_name} - ROC Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 ROC curve plot saved: {save_path} (AUC={roc_auc*100:.2f}%)")
    plt.close()


def plot_pr_curve(labels, probs, model_name, save_path):
    """Plot Precision-Recall curve and save with Average Precision (AP) annotation."""
    precision_vals, recall_vals, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)

    plt.figure(figsize=(8, 6))
    plt.plot(recall_vals, precision_vals, color='purple', lw=2, label=f'PR curve (AP = {ap*100:.2f}%)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'{model_name} - Precision-Recall Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 PR curve plot saved: {save_path} (AP={ap*100:.2f}%)")
    plt.close()


def plot_metrics_comparison(results, save_path):
    """Plot comparison of all models across different metrics."""
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
    model_names = list(results.keys())
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for i, (model_name, color) in enumerate(zip(model_names, colors)):
        values = [results[model_name][metric] * 100 for metric in metrics]
        offset = (i - len(model_names)/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=model_name.upper(), color=color, alpha=0.8)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%',
                   ha='center', va='bottom', fontsize=9)
    
    ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 105])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n📊 Model comparison plot saved: {save_path}")
    plt.close()


def plot_model_summary(results, save_path):
    """Create a comprehensive summary visualization."""
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    model_names = list(results.keys())
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    # 1. F1 Score comparison (main metric)
    ax1 = fig.add_subplot(gs[0, :])
    f1_scores = [results[m]['f1'] * 100 for m in model_names]
    bars = ax1.barh(model_names, f1_scores, color=colors, alpha=0.8)
    for i, (bar, score) in enumerate(zip(bars, f1_scores)):
        ax1.text(score + 0.5, i, f'{score:.2f}%', va='center', fontsize=11, fontweight='bold')
    ax1.set_xlabel('F1 Score (%)', fontsize=12, fontweight='bold')
    ax1.set_title('F1 Score Comparison (Primary Metric)', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.set_xlim([0, 105])
    
    # 2-5. Individual metric plots
    metrics = [('accuracy', 'Accuracy'), ('precision', 'Precision'), 
               ('recall', 'Recall'), ('f1', 'F1 Score')]
    positions = [(1, 0), (1, 1), (2, 0), (2, 1)]
    
    for (metric, title), pos in zip(metrics, positions):
        ax = fig.add_subplot(gs[pos])
        values = [results[m][metric] * 100 for m in model_names]
        bars = ax.bar(range(len(model_names)), values, color=colors, alpha=0.8)
        
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
        
        ax.set_xticks(range(len(model_names)))
        ax.set_xticklabels([m.upper() for m in model_names], rotation=45, ha='right')
        ax.set_ylabel('Score (%)', fontsize=10)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim([0, 105])
    
    plt.suptitle('Model Performance Summary', fontsize=16, fontweight='bold', y=0.995)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"📊 Model summary plot saved: {save_path}")
    plt.close()


# =============================================================================
# TRAINING UTILITIES
# =============================================================================

def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    """Train for one epoch with optional Mixed Precision."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        if not USE_XLA:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        
        # Mixed Precision (AMP) context
        if scaler is not None:
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            # Scale loss and backward
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            # Standard Fixed Precision
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            if USE_XLA and xm is not None:
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    if USE_XLA and xm is not None:
        xm.mark_step()
    
    epoch_loss = running_loss / max(1, total)
    epoch_acc = correct / max(1, total)
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            # Data is already on device if using TPU ParallelLoader
            if not USE_XLA:
                inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    # TPU-specific: Synchronize metrics
    if USE_XLA and xm is not None:
        xm.mark_step()
    
    val_loss = running_loss / total
    val_acc = correct / total
    return val_loss, val_acc


def evaluate_model(model, loader, device):
    """Comprehensive evaluation."""
    model.eval()
    all_preds = []
    all_probs = []  # For AUC calculation
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            # Data is already on device if using TPU ParallelLoader
            if not USE_XLA:
                inputs = inputs.to(device)
            
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)[:, 1]  # Probability of positive class (Attack)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy() if USE_XLA else labels.numpy())
    
    # TPU-specific: Synchronize before returning
    if USE_XLA and xm is not None:
        xm.mark_step()
    
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels
    }


def train_model(model_name, model, train_loader, val_loader, device, epochs=EPOCHS):
    """Complete training loop with early stopping and detailed logging."""
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*80}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Device: {device} {'(TPU Optimized)' if USE_XLA else ''}")
    print(f"Batch size: {BATCH_SIZE} {'(TPU optimized)' if USE_XLA else ''}")
    print(f"Optimizer: Adam (lr={LEARNING_RATE})")
    print(f"Loss: CrossEntropyLoss (weighted)")
    print(f"Early stopping patience: {PATIENCE} epochs")
    print(f"Max epochs: {epochs}")
    print("-" * 80)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    # Define ReduceLROnPlateau scheduler, LR: Learning Rate
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',           # Monitor validation loss (lower is better)
        factor=0.1,           # Reduce LR by 10% when triggered
        patience=5,           # Wait 5 epochs before reducing LR
        min_lr=1e-07,         # Minimum learning rate
        verbose=True          # Print message when LR is reduced
    )
    
    best_val_acc = 0.0
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    train_history = {'loss': [], 'acc': []}
    val_history = {'loss': [], 'acc': []}
    
    # Initialize GradScaler for AMP
    scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
    
    # Optimize model structure if PyTorch 2.0+
    if hasattr(torch, 'compile') and not USE_XLA:
        use_compile = True
        if device.type == 'cuda':
            cap = torch.cuda.get_device_capability(device)
            if cap[0] < 7:
                use_compile = False
                print(f"⚠️ torch.compile() disabled: GPU capability {cap[0]}.{cap[1]} < 7.0 (Triton limitation)")

        if use_compile:
            try:
                print("⚡ Using torch.compile() for faster training...")
                model = torch.compile(model)
            except Exception as e:
                print(f"⚠️ torch.compile failed: {e}")

    start_time = time.time()
    
    for epoch in range(epochs):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
        train_history['loss'].append(train_loss)
        train_history['acc'].append(train_acc)
        
        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        val_history['loss'].append(val_loss)
        val_history['acc'].append(val_acc)
        
        # Update learning rate scheduler (monitor val_loss)
        scheduler.step(val_loss)
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:6.2f}% | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:6.2f}% | "
                  f"{'✓ Best' if val_acc >= best_val_acc else ''}")
        
        # Early stopping logic (Monitor Loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            if (epoch + 1) % 5 != 0 and epoch != 0:
                print(f"Epoch {epoch+1:3d}/{epochs} | "
                      f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:6.2f}% | "
                      f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:6.2f}% | ✓ New Best (Loss)")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\n⚠️  Early stopping triggered at epoch {epoch+1}/{epochs}")
                print(f"    No improvement for {PATIENCE} consecutive epochs")
                print(f"    Best validation loss: {best_val_loss:.4f} (Acc: {best_val_acc*100:.2f}%)")
                break
    
    # Restore best weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✅  Restored best model weights")
    
    # TPU-specific: Ensure all operations are completed before saving
    if USE_XLA and xm is not None:
        xm.mark_step()
    
    elapsed = time.time() - start_time
    print(f"\n📊  Training Summary:")
    print(f"    Duration: {elapsed/60:.2f} minutes ({elapsed:.0f} seconds)")
    print(f"    Epochs completed: {epoch+1}/{epochs}")
    print(f"    Best validation accuracy: {best_val_acc*100:.2f}%")
    print(f"    Best validation loss: {best_val_loss:.4f}")
    print(f"    Final train accuracy: {train_history['acc'][-1]*100:.2f}%")
    
    return model, train_history, val_history


# =============================================================================
# MODEL TRAINING LOOP
# =============================================================================
print("\n" + "="*80)
print("MODEL POOL TRAINING")
print("="*80)

input_dim = X_train_scaled.shape[1]
output_dim = 2  # Binary classification

trained_models = {}
results = {}
training_histories = {}  # Store training histories for all models

# MLP Model
if MODELS_TO_TRAIN.get('mlp_model', False):
    model = MLPDropout(d_in=input_dim, d_out=output_dim).to(DEVICE)
    model, train_hist, val_hist = train_model(
        'MLP (Deep Multi-Layer Perceptron)', model, train_loader, val_loader, DEVICE, epochs=EPOCHS
    )
    trained_models['mlp'] = model
    training_histories['mlp'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    results['mlp'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'mlp_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Detailed results
    print(f"\n📊  MLP Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:  {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Precision: {eval_results['precision']*100:>6.2f}%")
    print(f"    Recall:    {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:  {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:   {eval_results['auc']*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(eval_results['labels'], eval_results['predictions'], 
                                target_names=['Benign', 'Attack'], digits=4))
    
    # Generate plots
    plot_training_history(train_hist, val_hist, 'MLP', 
                         os.path.join(PLOT_DIR, 'mlp_training_history.png'))
    plot_confusion_matrix(cm, 'MLP', 
                         os.path.join(PLOT_DIR, 'mlp_confusion_matrix.png'))
    # ROC and PR curves
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'MLP', os.path.join(PLOT_DIR, 'mlp_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'MLP', os.path.join(PLOT_DIR, 'mlp_pr_curve.png'))

# CNN Model
if MODELS_TO_TRAIN.get('cnn_model', False):
    model = DeepCNN(d_in=input_dim, d_out=output_dim).to(DEVICE)
    model, train_hist, val_hist = train_model(
        '1D-CNN (Convolutional Neural Network)', model, train_loader, val_loader, DEVICE, epochs=EPOCHS
    )
    trained_models['cnn'] = model
    training_histories['cnn'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    results['cnn'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'cnn_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Detailed results
    print(f"\n📊  CNN Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:  {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Precision: {eval_results['precision']*100:>6.2f}%")
    print(f"    Recall:    {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:  {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:   {eval_results['auc']*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(eval_results['labels'], eval_results['predictions'], 
                                target_names=['Benign', 'Attack'], digits=4))
    
    # Generate plots
    plot_training_history(train_hist, val_hist, 'CNN', 
                         os.path.join(PLOT_DIR, 'cnn_training_history.png'))
    plot_confusion_matrix(cm, 'CNN', 
                         os.path.join(PLOT_DIR, 'cnn_confusion_matrix.png'))
    # ROC and PR curves
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'CNN', os.path.join(PLOT_DIR, 'cnn_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'CNN', os.path.join(PLOT_DIR, 'cnn_pr_curve.png'))

# TCN Model
if MODELS_TO_TRAIN.get('tcn_model', False):
    model = DeepTCN(d_in=input_dim, d_out=output_dim).to(DEVICE)
    model, train_hist, val_hist = train_model(
        'TCN (Temporal Convolutional Network)', model, train_loader, val_loader, DEVICE, epochs=EPOCHS
    )
    trained_models['tcn'] = model
    training_histories['tcn'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    results['tcn'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'tcn_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Detailed results
    print(f"\n📊  TCN Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:  {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Precision: {eval_results['precision']*100:>6.2f}%")
    print(f"    Recall:    {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:  {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:   {eval_results['auc']*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(eval_results['labels'], eval_results['predictions'], 
                                target_names=['Benign', 'Attack'], digits=4))
    
    # Generate plots
    plot_training_history(train_hist, val_hist, 'TCN', 
                         os.path.join(PLOT_DIR, 'tcn_training_history.png'))
    plot_confusion_matrix(cm, 'TCN', 
                         os.path.join(PLOT_DIR, 'tcn_confusion_matrix.png'))
    # ROC and PR curves
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'TCN', os.path.join(PLOT_DIR, 'tcn_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'TCN', os.path.join(PLOT_DIR, 'tcn_pr_curve.png'))

# BiLSTM-Attention Model
if MODELS_TO_TRAIN.get('bilstm_model', False):
    model = BiLSTMAttention(d_in=input_dim, n_classes=output_dim).to(DEVICE)
    model, train_hist, val_hist = train_model(
        'BiLSTM-Attention (Bidirectional LSTM with Multi-Head Attention)', model, train_loader, val_loader, DEVICE, epochs=EPOCHS
    )
    trained_models['bilstm'] = model
    training_histories['bilstm'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    results['bilstm'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'bilstm_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Detailed results
    print(f"\n📊  BiLSTM-Attention Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:  {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Precision: {eval_results['precision']*100:>6.2f}%")
    print(f"    Recall:    {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:  {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:   {eval_results['auc']*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(eval_results['labels'], eval_results['predictions'], 
                                target_names=['Benign', 'Attack'], digits=4))
    
    # Generate plots
    plot_training_history(train_hist, val_hist, 'BiLSTM-Attention', 
                         os.path.join(PLOT_DIR, 'bilstm_training_history.png'))
    plot_confusion_matrix(cm, 'BiLSTM-Attention', 
                         os.path.join(PLOT_DIR, 'bilstm_confusion_matrix.png'))
    # ROC and PR curves
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'BiLSTM-Attention', os.path.join(PLOT_DIR, 'bilstm_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'BiLSTM-Attention', os.path.join(PLOT_DIR, 'bilstm_pr_curve.png'))

# =============================================================================
# VISUALIZATION SUMMARY
# =============================================================================
if results:
    print("\n" + "="*80)
    print("GENERATING COMPARISON VISUALIZATIONS")
    print("="*80)
    
    # Create comparison plots
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison.png'))
    plot_model_summary(results, os.path.join(PLOT_DIR, 'model_summary.png'))
    
    print(f"\n✅ All plots saved to: {PLOT_DIR}")
    print(f"   - Individual training histories ({len(training_histories)} models)")
    print(f"   - Individual confusion matrices ({len(results)} models)")
    print(f"   - Model comparison plot")
    print(f"   - Comprehensive model summary plot")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)

if results:
    print("\nModel Comparison:")
    print("-" * 80)
    print(f"{'Model':<15} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'AUC-ROC':<12}")
    print("-" * 80)
    
    for model_name, res in results.items():
        print(f"{model_name.upper():<15} "
              f"{res['accuracy']*100:>10.2f}% "
              f"{res['precision']*100:>11.2f}% "
              f"{res['recall']*100:>10.2f}% "
              f"{res['f1']*100:>11.2f}% "
              f"{res['auc']*100:>10.2f}%")
    
    print("-" * 80)
    
    # Find best model
    best_model_name = max(results, key=lambda k: results[k]['f1'])
    best_f1 = results[best_model_name]['f1']
    print(f"\n🏆 Best Model: {best_model_name.upper()} (F1: {best_f1*100:.2f}%)")
    
    print(f"\n✅ All models saved to: {OUT_DIR}")
else:
    print("\n⚠️ No models were trained. Check MODELS_TO_TRAIN configuration.")

# =============================================================================
# ENSEMBLE EVALUATION (Baseline - Static Mean Logits)
# =============================================================================
if len(trained_models) > 1:
    print("\n" + "="*80)
    print("ENSEMBLE EVALUATION (BASELINE)")
    print("="*80)
    print("Strategy: Static Mean Logits (all models weighted equally)")
    print(f"Models in ensemble: {list(trained_models.keys())}")
    print("-" * 80)
    
    # Evaluate ensemble
    all_preds = []
    all_probs = []
    all_labels = []
    
    for model in trained_models.values():
        model.eval()
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            if not USE_XLA:
                inputs = inputs.to(DEVICE)
            
            # Collect logits from all models
            logits_list = []
            for model in trained_models.values():
                logits = model(inputs)
                logits_list.append(logits)
            
            # Mean logits ensemble
            ensemble_logits = torch.stack(logits_list, dim=0).mean(dim=0)
            probs = F.softmax(ensemble_logits, dim=1)[:, 1]
            _, predicted = torch.max(ensemble_logits.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy() if USE_XLA else labels.numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    ensemble_accuracy = accuracy_score(all_labels, all_preds)
    ensemble_precision = precision_score(all_labels, all_preds, zero_division=0)
    ensemble_recall = recall_score(all_labels, all_preds, zero_division=0)
    ensemble_f1 = f1_score(all_labels, all_preds, zero_division=0)
    ensemble_auc = roc_auc_score(all_labels, all_probs)
    
    print(f"\n📊 Ensemble Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:  {ensemble_accuracy*100:>6.2f}%")
    print(f"    Precision: {ensemble_precision*100:>6.2f}%")
    print(f"    Recall:    {ensemble_recall*100:>6.2f}%")
    print(f"    F1 Score:  {ensemble_f1*100:>6.2f}%")
    print(f"    AUC-ROC:   {ensemble_auc*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    ensemble_cm = confusion_matrix(all_labels, all_preds)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {ensemble_cm[0,0]:>6,}  {ensemble_cm[0,1]:>6,}")
    print(f"           Attack  {ensemble_cm[1,0]:>6,}  {ensemble_cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(all_labels, all_preds, 
                                target_names=['Benign', 'Attack'], digits=4))
    
    # Add to results for comparison
    results['ensemble'] = {
        'accuracy': ensemble_accuracy,
        'precision': ensemble_precision,
        'recall': ensemble_recall,
        'f1': ensemble_f1,
        'auc': ensemble_auc,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels
    }
    
    # Generate ensemble plots
    plot_confusion_matrix(ensemble_cm, 'Ensemble (Mean Logits)', 
                         os.path.join(PLOT_DIR, 'ensemble_confusion_matrix.png'))
    plot_roc_curve(all_labels, all_probs, 'Ensemble (Mean Logits)', 
                  os.path.join(PLOT_DIR, 'ensemble_roc_curve.png'))
    plot_pr_curve(all_labels, all_probs, 'Ensemble (Mean Logits)', 
                 os.path.join(PLOT_DIR, 'ensemble_pr_curve.png'))
    
    # Update comparison plots with ensemble
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison_with_ensemble.png'))
    
    # Print improvement over best single model
    if best_f1 < ensemble_f1:
        improvement = (ensemble_f1 - best_f1) * 100
        print(f"\n🎯 Ensemble improvement: +{improvement:.2f}% F1 over best single model")
    else:
        degradation = (best_f1 - ensemble_f1) * 100
        print(f"\n⚠️ Ensemble degradation: -{degradation:.2f}% F1 vs best single model")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80 + "\n")







HARDWARE DETECTION
PyTorch: 2.6.0+cu124
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
⚠️ TPU not available (torch_xla not installed)
   For Kaggle TPU: Runtime → Change runtime type → TPU

✅ CUDA Available
   GPU Count: 1
   GPU 0: Tesla P100-PCIE-16GB (17.06 GB)

🎯 Using CUDA: cuda


DATA LOADING AND PREPROCESSING
Loading 7 CSV files from /kaggle/input/cic-ids2017/MachineLearningCVE...
  [1/7] Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
      Initial shape: (225745, 79)
      Memory usage: 147.66 MB
  [2/7] Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
      Initial shape: (286467, 79)
      Memory usage: 187.99 MB
  [3/7] Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
      Initial shape: (191033, 79)
      Memory usage: 125.15 MB
  [4/7] Loading: Monday-WorkingHours.pcap_ISCX.csv
      Initial shape: (529918, 79)
      Memory usage: 347.19 MB
  [5/7] Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
      Initial shape: (288602,

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3086463810.py:873: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3086463810.py:728: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


⚠️ torch.compile() disabled: GPU capability 6.0 < 7.0 (Triton limitation)
Epoch   1/100 | Train Loss: 0.0999 Acc:  96.89% | Val Loss: 0.0388 Acc:  98.65% | ✓ Best
Epoch   2/100 | Train Loss: 0.0399 Acc:  98.58% | Val Loss: 0.0313 Acc:  99.05% | ✓ New Best (Loss)
Epoch   3/100 | Train Loss: 0.0326 Acc:  98.91% | Val Loss: 0.0267 Acc:  99.25% | ✓ New Best (Loss)
Epoch   4/100 | Train Loss: 0.0293 Acc:  99.05% | Val Loss: 0.0241 Acc:  99.28% | ✓ New Best (Loss)
Epoch   5/100 | Train Loss: 0.0272 Acc:  99.16% | Val Loss: 0.0240 Acc:  99.38% | ✓ Best
Epoch   6/100 | Train Loss: 0.0260 Acc:  99.20% | Val Loss: 0.0215 Acc:  99.38% | ✓ New Best (Loss)
Epoch   7/100 | Train Loss: 0.0248 Acc:  99.25% | Val Loss: 0.0206 Acc:  99.46% | ✓ New Best (Loss)
Epoch   9/100 | Train Loss: 0.0228 Acc:  99.32% | Val Loss: 0.0198 Acc:  99.42% | ✓ New Best (Loss)
Epoch  10/100 | Train Loss: 0.0223 Acc:  99.34% | Val Loss: 0.0204 Acc:  99.31% | 
Epoch  11/100 | Train Loss: 0.0238 Acc:  99.28% | Val Loss: 0.019

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3086463810.py:873: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3086463810.py:728: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch   1/100 | Train Loss: 0.1003 Acc:  95.88% | Val Loss: 0.0598 Acc:  96.80% | ✓ Best
Epoch   2/100 | Train Loss: 0.0357 Acc:  98.71% | Val Loss: 0.0361 Acc:  98.28% | ✓ New Best (Loss)
Epoch   3/100 | Train Loss: 0.0291 Acc:  99.00% | Val Loss: 0.0357 Acc:  98.29% | ✓ New Best (Loss)
Epoch   5/100 | Train Loss: 0.0258 Acc:  99.14% | Val Loss: 0.0484 Acc:  97.46% | 
Epoch   6/100 | Train Loss: 0.0245 Acc:  99.23% | Val Loss: 0.0287 Acc:  98.69% | ✓ New Best (Loss)
Epoch   9/100 | Train Loss: 0.0220 Acc:  99.34% | Val Loss: 0.0260 Acc:  99.54% | ✓ New Best (Loss)
Epoch  10/100 | Train Loss: 0.0215 Acc:  99.33% | Val Loss: 0.0271 Acc:  98.73% | 
Epoch  12/100 | Train Loss: 0.0210 Acc:  99.35% | Val Loss: 0.0250 Acc:  98.84% | ✓ New Best (Loss)
Epoch  15/100 | Train Loss: 0.0209 Acc:  99.36% | Val Loss: 0.0437 Acc:  99.67% | ✓ Best
Epoch  17/100 | Train Loss: 0.0206 Acc:  99.35% | Val Loss: 0.0184 Acc:  99.47% | ✓ New Best (Loss)
Epoch  20/100 | Train Loss: 0.0195 Acc:  99.41% | Val Lo

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3086463810.py:873: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3086463810.py:728: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch   1/100 | Train Loss: 0.1067 Acc:  94.51% | Val Loss: 0.0431 Acc:  97.66% | ✓ Best
Epoch   3/100 | Train Loss: 0.0313 Acc:  98.93% | Val Loss: 0.0387 Acc:  99.68% | ✓ New Best (Loss)
Epoch   5/100 | Train Loss: 0.0272 Acc:  99.12% | Val Loss: 0.0343 Acc:  99.38% | 
Epoch   6/100 | Train Loss: 0.0247 Acc:  99.23% | Val Loss: 0.0279 Acc:  98.56% | ✓ New Best (Loss)
Epoch   8/100 | Train Loss: 0.0235 Acc:  99.30% | Val Loss: 0.0216 Acc:  99.36% | ✓ New Best (Loss)
Epoch  10/100 | Train Loss: 0.0236 Acc:  99.28% | Val Loss: 0.1693 Acc:  91.29% | 
Epoch  11/100 | Train Loss: 0.0226 Acc:  99.28% | Val Loss: 0.0214 Acc:  99.07% | ✓ New Best (Loss)
Epoch  15/100 | Train Loss: 0.0207 Acc:  99.39% | Val Loss: 0.0335 Acc:  98.39% | 
Epoch  18/100 | Train Loss: 0.0175 Acc:  99.47% | Val Loss: 0.0163 Acc:  99.46% | ✓ New Best (Loss)
Epoch  20/100 | Train Loss: 0.0167 Acc:  99.50% | Val Loss: 0.0165 Acc:  99.47% | ✓ Best
Epoch  22/100 | Train Loss: 0.0165 Acc:  99.52% | Val Loss: 0.0162 Acc:  

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3086463810.py:873: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3086463810.py:728: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch   1/100 | Train Loss: 0.0723 Acc:  97.09% | Val Loss: 0.0437 Acc:  98.70% | ✓ Best
Epoch   2/100 | Train Loss: 0.0425 Acc:  98.36% | Val Loss: 0.0324 Acc:  99.12% | ✓ New Best (Loss)
Epoch   3/100 | Train Loss: 0.0374 Acc:  98.67% | Val Loss: 0.0265 Acc:  99.29% | ✓ New Best (Loss)
Epoch   4/100 | Train Loss: 0.0322 Acc:  98.95% | Val Loss: 0.0262 Acc:  99.22% | ✓ New Best (Loss)
Epoch   5/100 | Train Loss: 0.0309 Acc:  98.96% | Val Loss: 0.0251 Acc:  99.29% | ✓ Best
Epoch   7/100 | Train Loss: 0.0274 Acc:  99.14% | Val Loss: 0.0229 Acc:  99.29% | ✓ New Best (Loss)
Epoch  10/100 | Train Loss: 0.0256 Acc:  99.20% | Val Loss: 0.0212 Acc:  99.33% | ✓ Best
Epoch  15/100 | Train Loss: 0.0227 Acc:  99.31% | Val Loss: 0.0194 Acc:  99.36% | ✓ Best
Epoch  17/100 | Train Loss: 0.0235 Acc:  99.26% | Val Loss: 0.0180 Acc:  99.43% | ✓ New Best (Loss)
Epoch  20/100 | Train Loss: 0.0216 Acc:  99.33% | Val Loss: 0.0190 Acc:  99.39% | 
Epoch  22/100 | Train Loss: 0.0221 Acc:  99.28% | Val Loss: 0